**1. How does the frequency of non-breaking errors correlate with catastrophic component failures?**

In [0]:
# Question 1: Analyze errors, failures, and errors occurring 24–48 hours before failure

from pyspark.sql import functions as F
silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

errors = spark.read.format("delta").load(f"{silver_path}/PdM_errors")
failures = spark.read.format("delta").load(f"{silver_path}/PdM_failures")

# Show overall error and failure frequencies
print("===== Error Frequency =====")
display(errors.groupBy("errorID").count().orderBy("count", ascending=False))

print("===== Failure Frequency =====")
display(failures.groupBy("failure").count().orderBy("count", ascending=False))

# Find errors occurring 24–48 hours before a failure
pre_failure = (
    errors.alias("e")
    .join(
        failures.alias("f"),
        (F.col("e.machineID") == F.col("f.machineID")) &
        (F.col("e.datetime") < F.col("f.datetime")) &
        (F.col("e.datetime") >= F.col("f.datetime") - F.expr("INTERVAL 48 HOURS")) &
        (F.col("e.datetime") <= F.col("f.datetime") - F.expr("INTERVAL 24 HOURS")),
        "inner"
    )
    .select("e.errorID")
)

# Compare total errors with pre-failure errors
result = (
    errors.groupBy("errorID").count()
    .withColumnRenamed("count", "total_errors")
    .join(
        pre_failure.groupBy("errorID").count()
        .withColumnRenamed("count", "pre_failure_errors"),
        "errorID",
        "left"
    )
    .fillna(0)
    .withColumn(
        "pre_failure_percentage",
        F.round(F.col("pre_failure_errors") / F.col("total_errors") * 100, 2)
    )
    .orderBy("pre_failure_percentage", ascending=False)
)

display(result)

===== Error Frequency =====


errorID,count
error1,1010
error2,988
error3,838
error4,727
error5,356


===== Failure Frequency =====


failure,count
comp2,259
comp1,192
comp4,179
comp3,131


errorID,total_errors,pre_failure_errors,pre_failure_percentage
error5,356,204,57.3
error3,838,302,36.04
error2,988,294,29.76
error1,1010,224,22.18
error4,727,148,20.36


In [0]:
# Saving results as a Gold Delta table

result.write.format("delta").save(
    f"{gold_path}/error_pre_failure_analysis"
)

**2. What is the typical relationship between the age/model of a machine and its baseline telemetry metrics?**

In [0]:
# Question 2: Compare machine age and model with baseline telemetry

from pyspark.sql import functions as F

silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

machines = spark.read.format("delta").load(f"{silver_path}/PdM_machines")
telemetry = spark.read.format("delta").load(f"{silver_path}/PdM_telemetry")

# Calculate average telemetry values for each machine
machine_telemetry = (
    telemetry
    .groupBy("machineID")
    .agg(
        F.round(F.avg("volt"), 2).alias("avg_voltage"),
        F.round(F.avg("rotate"), 2).alias("avg_rotation"),
        F.round(F.avg("pressure"), 2).alias("avg_pressure"),
        F.round(F.avg("vibration"), 2).alias("avg_vibration")
    )
)

# Join machine age and model with telemetry averages
result = (
    machines
    .join(machine_telemetry, "machineID")
    .select(
        "machineID",
        "model",
        "age",
        "avg_voltage",
        "avg_rotation",
        "avg_pressure",
        "avg_vibration"
    )
)

display(result.limit(20))

machineID,model,age,avg_voltage,avg_rotation,avg_pressure,avg_vibration
12,model3,9,170.94,447.24,100.57,40.3
50,model4,4,170.89,445.71,100.81,40.32
56,model1,10,170.91,446.91,101.3,40.3
73,model2,20,171.0,448.41,101.5,40.51
83,model4,18,170.9,445.47,100.82,40.64
93,model3,18,170.78,446.58,100.81,40.46
97,model2,14,170.34,447.03,101.04,40.55
20,model2,16,170.95,446.56,101.31,40.57
61,model4,2,170.65,446.91,100.5,40.32
71,model2,18,170.9,446.55,101.3,40.48


In [0]:
# Compare baseline telemetry across machine age and model

age_summary = (
    result
    .groupBy("age")
    .agg(
        F.round(F.avg("avg_voltage"), 2).alias("avg_voltage"),
        F.round(F.avg("avg_rotation"), 2).alias("avg_rotation"),
        F.round(F.avg("avg_pressure"), 2).alias("avg_pressure"),
        F.round(F.avg("avg_vibration"), 2).alias("avg_vibration")
    )
    .orderBy("age")
)

model_summary = (
    result
    .groupBy("model")
    .agg(
        F.round(F.avg("avg_voltage"), 2).alias("avg_voltage"),
        F.round(F.avg("avg_rotation"), 2).alias("avg_rotation"),
        F.round(F.avg("avg_pressure"), 2).alias("avg_pressure"),
        F.round(F.avg("avg_vibration"), 2).alias("avg_vibration")
    )
    .orderBy("model")
)

print("===== Telemetry by Machine Age =====")
display(age_summary)

print("===== Telemetry by Machine Model =====")
display(model_summary)

===== Telemetry by Machine Age =====


age,avg_voltage,avg_rotation,avg_pressure,avg_vibration
0,170.7,446.58,100.76,40.24
1,170.8,445.96,100.56,40.31
2,170.77,446.82,100.99,40.25
3,170.81,446.55,100.87,40.26
4,170.81,446.58,100.83,40.28
5,170.89,446.75,100.77,40.29
6,170.84,446.4,100.75,40.27
7,170.72,446.69,100.75,40.31
8,170.66,446.58,100.65,40.48
9,170.9,446.71,100.68,40.26


===== Telemetry by Machine Model =====


model,avg_voltage,avg_rotation,avg_pressure,avg_vibration
model1,170.78,446.37,101.27,40.44
model2,170.72,446.81,101.22,40.39
model3,170.8,446.51,100.67,40.4
model4,170.77,446.71,100.67,40.34


In [0]:
# Save Question 2 results as Gold Delta tables

result.write.format("delta").save(f"{gold_path}/machine_telemetry_baseline")

age_summary.write.format("delta").save(f"{gold_path}/telemetry_by_age")

model_summary.write.format("delta").save(f"{gold_path}/telemetry_by_model")

**3. Does the time elapsed since the last scheduled maintenance impact the stability of sensor readings?**

In [0]:
# Question 3: Calculate sensor stability based on time since maintenance

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

maint = spark.read.format("delta").load(f"{silver_path}/PdM_maint")
telemetry = spark.read.format("delta").load(f"{silver_path}/PdM_telemetry")

# Find the most recent maintenance event for each machine before each telemetry reading
window = Window.partitionBy("machineID").orderBy("datetime")

maintenance_times = maint.select(
    "machineID",
    F.col("datetime").alias("maint_time")
)

sensor_data = telemetry.alias("t").join(
    maintenance_times.alias("m"),
    (F.col("t.machineID") == F.col("m.machineID")) &
    (F.col("m.maint_time") <= F.col("t.datetime")),
    "left"
)

# Keep the latest maintenance time before each telemetry reading
sensor_data = (
    sensor_data
    .groupBy(
        "t.machineID",
        "t.datetime",
        "t.volt",
        "t.rotate",
        "t.pressure",
        "t.vibration"
    )
    .agg(F.max("m.maint_time").alias("last_maintenance"))
    .withColumn(
        "hours_since_maintenance",
        (F.col("datetime").cast("long") - F.col("last_maintenance").cast("long")) / 3600
    )
)

display(sensor_data.limit(20))

machineID,datetime,volt,rotate,pressure,vibration,last_maintenance,hours_since_maintenance
1,2015-07-03T00:00:00.000Z,167.70546725457,501.773234550845,94.3470220083674,39.9639952349927,2015-06-19T06:00:00.000Z,330.0
1,2015-08-05T19:00:00.000Z,194.939374101986,532.882562534404,106.490381498666,33.0019262817164,2015-08-03T06:00:00.000Z,61.0
1,2015-08-14T15:00:00.000Z,158.832451887049,428.924016335967,99.0821538821098,37.783641572471,2015-08-03T06:00:00.000Z,273.0
1,2015-09-04T09:00:00.000Z,175.900223587957,467.86872428576,104.617164122189,41.5466739736302,2015-09-02T06:00:00.000Z,51.0
1,2015-09-13T05:00:00.000Z,193.4582467685,309.84678769201,84.3860318947616,35.7541905162606,2015-09-02T06:00:00.000Z,263.0
1,2015-11-27T06:00:00.000Z,200.373241343855,498.144331145875,99.3929451692545,52.808931810771,2015-11-16T06:00:00.000Z,264.0
1,2015-12-02T13:00:00.000Z,162.178795603216,315.628919999962,128.741462284293,36.0466442166604,2015-12-01T06:00:00.000Z,31.0
1,2015-12-07T02:00:00.000Z,170.99155081963,449.943703295256,91.8564197841445,45.4154018945394,2015-12-01T06:00:00.000Z,140.0
2,2015-12-25T20:00:00.000Z,190.216778909077,461.837034273397,103.192600044696,39.8644556910066,2015-12-14T06:00:00.000Z,278.0
3,2015-03-24T04:00:00.000Z,165.800915127079,407.842465007105,86.361856222935,43.2364555885165,2015-03-23T06:00:00.000Z,22.0


In [0]:
# Compare sensor fluctuations at different times after maintenance

result = (
    sensor_data
    .filter(F.col("last_maintenance").isNotNull())
    .withColumn(
        "maintenance_period",
        F.when(F.col("hours_since_maintenance") <= 24, "0-24 hours")
         .when(F.col("hours_since_maintenance") <= 72, "24-72 hours")
         .when(F.col("hours_since_maintenance") <= 168, "3-7 days")
         .otherwise("More than 7 days")
    )
    .groupBy("maintenance_period")
    .agg(
        F.round(F.avg("volt"), 2).alias("avg_voltage"),
        F.round(F.stddev("volt"), 2).alias("voltage_fluctuation"),
        F.round(F.avg("rotate"), 2).alias("avg_rotation"),
        F.round(F.stddev("rotate"), 2).alias("rotation_fluctuation"),
        F.round(F.avg("pressure"), 2).alias("avg_pressure"),
        F.round(F.stddev("pressure"), 2).alias("pressure_fluctuation"),
        F.round(F.avg("vibration"), 2).alias("avg_vibration"),
        F.round(F.stddev("vibration"), 2).alias("vibration_fluctuation")
    )
    .orderBy("maintenance_period")
)

display(result)

maintenance_period,avg_voltage,voltage_fluctuation,avg_rotation,rotation_fluctuation,avg_pressure,pressure_fluctuation,avg_vibration,vibration_fluctuation
0-24 hours,170.56,15.42,447.6,51.84,100.69,10.92,40.31,5.25
24-72 hours,170.53,15.28,447.49,51.75,100.69,10.86,40.23,5.25
3-7 days,170.65,15.45,447.96,51.65,100.63,10.78,40.34,5.33
More than 7 days,170.9,15.59,445.79,53.32,101.0,11.2,40.44,5.42


In [0]:
# Save Question 3 results as a Gold Delta table

result.write.format("delta").save(
    f"{gold_path}/maintenance_sensor_stability"
)

**4. Can we forecast telemetry sensor values (e.g., voltage or vibration) 12 hours into the future to preemptively catch anomalies?**


In [0]:
# Question 4: Prepare telemetry data for 12-hour LSTM forecasting

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

telemetry = spark.read.format("delta").load(
    "/Volumes/workspace/default/t_series_forecasting_data/silver_data/PdM_telemetry"
)

# Select one machine and order its telemetry by time
machine_df = (
    telemetry
    .filter("machineID = 1")
    .select("datetime", "volt", "vibration")
    .orderBy("datetime")
)

pdf = machine_df.toPandas()

# Scale voltage and vibration for LSTM training
scaler = MinMaxScaler()
data = scaler.fit_transform(pdf[["volt", "vibration"]])

# Create sequences using the previous 24 hours to predict the next 12 hours
X, y = [], []

for i in range(24, len(data) - 12, 12):
    X.append(data[i-24:i])
    y.append(data[i:i+12])

X = np.array(X)
y = np.array(y)

print("Training samples:", X.shape)
print("Target shape:", y.shape)

Training samples: (728, 24, 2)
Target shape: (728, 12, 2)


In [0]:
# Train the LSTM model and forecast voltage and vibration for the next 12 hours

model = Sequential([
    LSTM(64, input_shape=(X.shape[1], X.shape[2])),
    Dense(24)
])

model.compile(optimizer="adam", loss="mse")

model.fit(
    X,
    y.reshape(y.shape[0], -1),
    epochs=10,
    batch_size=32,
    verbose=1
)

# Use the latest 24 hours to forecast the next 12 hours
last_sequence = data[-24:].reshape(1, 24, 2)

forecast_scaled = model.predict(last_sequence, verbose=0)

forecast = scaler.inverse_transform(
    forecast_scaled.reshape(12, 2)
)

forecast_df = pd.DataFrame(
    forecast,
    columns=["forecast_voltage", "forecast_vibration"]
)

display(forecast_df)

Epoch 1/10


/local_disk0/.ephemeral_nfs/envs/pythonEnv-473452b3-5864-4ddf-8301-359c23c41198/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0855
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0180
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0160
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0158
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0158
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0159
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0158
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0158
Epoch 9/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0158
Epoch 10/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0157


forecast_voltage,forecast_vibration
171.81998,40.42657
171.516,40.37932
169.97203,39.636803
172.15076,39.167484
168.95747,39.342068
169.25038,40.01828
171.09438,39.930008
170.2227,40.499218
168.61118,39.856045
170.29214,39.09036


In [0]:
# Check whether the 12-hour forecast crosses defined safety thresholds

VOLTAGE_THRESHOLD = 180
VIBRATION_THRESHOLD = 50

forecast_df["alert"] = np.where(
    (forecast_df["forecast_voltage"] > VOLTAGE_THRESHOLD) |
    (forecast_df["forecast_vibration"] > VIBRATION_THRESHOLD),
    "ALERT",
    "SAFE"
)

print("===== 12-Hour Forecast =====")
display(forecast_df)

print("===== Alert Summary =====")
print(
    "ALERT"
    if (forecast_df["alert"] == "ALERT").any()
    else "SAFE"
)

===== 12-Hour Forecast =====


forecast_voltage,forecast_vibration,alert
171.81998,40.42657,SAFE
171.516,40.37932,SAFE
169.97203,39.636803,SAFE
172.15076,39.167484,SAFE
168.95747,39.342068,SAFE
169.25038,40.01828,SAFE
171.09438,39.930008,SAFE
170.2227,40.499218,SAFE
168.61118,39.856045,SAFE
170.29214,39.09036,SAFE


===== Alert Summary =====
SAFE


In [0]:
# Save Question 4 forecast as a Gold Delta table

forecast_spark = spark.createDataFrame(forecast_df)

forecast_spark.write.format("delta").mode("overwrite").save(
    f"{gold_path}/telemetry_12h_forecast"
)

display(forecast_spark)

forecast_voltage,forecast_vibration,alert
171.81998,40.42657,SAFE
171.516,40.37932,SAFE
169.97203,39.636803,SAFE
172.15076,39.167484,SAFE
168.95747,39.342068,SAFE
169.25038,40.01828,SAFE
171.09438,39.930008,SAFE
170.2227,40.499218,SAFE
168.61118,39.856045,SAFE
170.29214,39.09036,SAFE


**5. How can we identify multi-sensor anomalous states that do not trigger standard error codes?**

In [0]:
# Question 5: Detect multi-sensor anomalies using Isolation Forest

from pyspark.sql import functions as F
from sklearn.ensemble import IsolationForest
import pandas as pd

silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

telemetry = spark.read.format("delta").load(
    f"{silver_path}/PdM_telemetry"
)

# Use sensor values for anomaly detection
sensor_df = (
    telemetry
    .select(
        "datetime",
        "machineID",
        "volt",
        "rotate",
        "pressure",
        "vibration"
    )
    .dropna()
)

pdf = sensor_df.toPandas()

print("Total records:", len(pdf))
display(pdf.head())

Total records: 876100


datetime,machineID,volt,rotate,pressure,vibration
2015-01-01T16:00:00.000Z,1,167.060980719256,382.483542906686,103.780662505568,42.6757999060571
2015-01-03T02:00:00.000Z,1,177.70644167607,521.837936384375,101.528033099946,35.4684440422458
2015-01-03T17:00:00.000Z,1,164.148094555294,429.822143515459,104.532123632465,43.7886815487723
2015-01-04T02:00:00.000Z,1,165.187401310127,475.126936759955,114.584420977964,57.5490157337645
2015-01-04T11:00:00.000Z,1,215.656488291429,458.097746169369,95.0362798574919,51.6479808086499


In [0]:
# Train Isolation Forest

features = ["volt", "rotate", "pressure", "vibration"]

model = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42
)

pdf["prediction"] = model.fit_predict(pdf[features])

pdf["anomaly"] = pdf["prediction"].apply(
    lambda x: "ANOMALY" if x == -1 else "NORMAL"
)

anomalies = pdf[pdf["anomaly"] == "ANOMALY"]

print("===== Anomaly Summary =====")
print("Total records:", len(pdf))
print("Anomalies:", len(anomalies))

display(anomalies.head(20))

===== Anomaly Summary =====
Total records: 876100
Anomalies: 8761


datetime,machineID,volt,rotate,pressure,vibration,prediction,anomaly
2015-03-17T17:00:00.000Z,1,189.111392198928,593.774867091016,78.9268034463212,45.3309084422527,-1,ANOMALY
2015-06-03T20:00:00.000Z,1,142.57410116796,318.864769296372,96.1468034918873,51.6456986090519,-1,ANOMALY
2015-02-06T14:00:00.000Z,2,211.907729127498,491.397989691266,120.257596360816,46.7546037363671,-1,ANOMALY
2015-03-02T15:00:00.000Z,2,195.243813587574,508.687330382525,75.043984071194,33.5751003430453,-1,ANOMALY
2015-07-09T03:00:00.000Z,2,210.157480517918,473.368184987732,81.6894446436804,50.2492607264404,-1,ANOMALY
2015-09-22T13:00:00.000Z,2,183.775472165953,557.564815679449,139.967168245863,48.091586298747,-1,ANOMALY
2015-10-05T00:00:00.000Z,2,220.734969672089,392.091051376846,124.412352098287,35.5655654594411,-1,ANOMALY
2015-10-29T02:00:00.000Z,2,142.792574218094,342.347198619575,87.9973664514401,52.18718333442,-1,ANOMALY
2015-11-27T20:00:00.000Z,2,225.366225624189,484.646343610966,76.6581293141637,41.1504294925166,-1,ANOMALY
2015-05-20T09:00:00.000Z,3,151.508205188042,326.504883119741,97.0556300334492,26.9605558018187,-1,ANOMALY


In [0]:
# Save Question 5 anomaly results

anomaly_spark = spark.createDataFrame(
    anomalies[
        ["datetime", "machineID", "volt", "rotate",
         "pressure", "vibration", "anomaly"]
    ]
)

anomaly_spark.write.format("delta").mode("overwrite").save(
    f"{gold_path}/multi_sensor_anomalies"
)

display(anomaly_spark.limit(20))

datetime,machineID,volt,rotate,pressure,vibration,anomaly
2015-03-17T17:00:00.000Z,1,189.111392198928,593.774867091016,78.9268034463212,45.3309084422527,ANOMALY
2015-06-03T20:00:00.000Z,1,142.57410116796,318.864769296372,96.1468034918873,51.6456986090519,ANOMALY
2015-02-06T14:00:00.000Z,2,211.907729127498,491.397989691266,120.257596360816,46.7546037363671,ANOMALY
2015-03-02T15:00:00.000Z,2,195.243813587574,508.687330382525,75.043984071194,33.5751003430453,ANOMALY
2015-07-09T03:00:00.000Z,2,210.157480517918,473.368184987732,81.6894446436804,50.2492607264404,ANOMALY
2015-09-22T13:00:00.000Z,2,183.775472165953,557.564815679449,139.967168245863,48.091586298747,ANOMALY
2015-10-05T00:00:00.000Z,2,220.734969672089,392.091051376846,124.412352098287,35.5655654594411,ANOMALY
2015-10-29T02:00:00.000Z,2,142.792574218094,342.347198619575,87.9973664514401,52.18718333442,ANOMALY
2015-11-27T20:00:00.000Z,2,225.366225624189,484.646343610966,76.6581293141637,41.1504294925166,ANOMALY
2015-05-20T09:00:00.000Z,3,151.508205188042,326.504883119741,97.0556300334492,26.9605558018187,ANOMALY


**6. What is the probability that a specific machine will fail within the next 24 hours given its current 7-day rolling window of sensor data?**

In [0]:
# Q6: Create 7-day telemetry features and failure target

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

telemetry = spark.read.format("delta").load(
    f"{silver_path}/PdM_telemetry"
)

failures = spark.read.format("delta").load(
    f"{silver_path}/PdM_failures"
)

# Convert timestamp to seconds
telemetry = telemetry.withColumn(
    "timestamp_long",
    F.col("datetime").cast("long")
)

# 7-day rolling window
window_7d = (
    Window
    .partitionBy("machineID")
    .orderBy("timestamp_long")
    .rangeBetween(-7 * 24 * 3600, 0)
)

# Calculate rolling features
features = (
    telemetry
    .withColumn("avg_volt_7d", F.avg("volt").over(window_7d))
    .withColumn("avg_rotate_7d", F.avg("rotate").over(window_7d))
    .withColumn("avg_pressure_7d", F.avg("pressure").over(window_7d))
    .withColumn("avg_vibration_7d", F.avg("vibration").over(window_7d))
    .withColumn("std_volt_7d", F.stddev("volt").over(window_7d))
    .withColumn("std_rotate_7d", F.stddev("rotate").over(window_7d))
    .withColumn("std_pressure_7d", F.stddev("pressure").over(window_7d))
    .withColumn("std_vibration_7d", F.stddev("vibration").over(window_7d))
)

# Create failure target
target = (
    features.alias("t")
    .join(
        failures.alias("f"),
        (F.col("t.machineID") == F.col("f.machineID")) &
        (F.col("f.datetime") > F.col("t.datetime")) &
        (F.col("f.datetime") <= F.col("t.datetime") + F.expr("INTERVAL 24 HOURS")),
        "left"
    )
    .withColumn(
        "failure_next_24h",
        F.when(F.col("f.datetime").isNotNull(), 1).otherwise(0)
    )
    .select("t.*", "failure_next_24h")
    .dropDuplicates(["machineID", "datetime"])
    .dropna()
)

print("Total records:", target.count())

display(target.limit(20))

Total records: 876000


datetime,machineID,volt,rotate,pressure,vibration,timestamp_long,avg_volt_7d,avg_rotate_7d,avg_pressure_7d,avg_vibration_7d,std_volt_7d,std_rotate_7d,std_pressure_7d,std_vibration_7d,failure_next_24h
2015-01-06T06:00:00.000Z,12,192.159514208444,400.696465608481,121.807238776055,48.125120930802,1420524000,171.72643875032895,435.33371087514183,99.79511806406099,40.429909274342485,17.73075812888031,65.37574602220555,10.318377493457197,5.245893724923236,1
2015-01-06T07:00:00.000Z,12,214.469540017086,419.695742687293,95.1928274598296,37.4608596299477,1420527600,172.07679203940074,435.2055308080283,99.7573943705837,40.40557280184745,18.07641529471984,65.1204301852163,10.2840955419794,5.231082498135742,1
2015-01-06T08:00:00.000Z,12,215.050520620208,299.201196745852,95.4553529120395,43.7551480571887,1420531200,172.42617194656177,434.0998045148399,99.72241842376627,40.43280512099656,18.414467487972757,66.00223028049632,10.249203986620493,5.2183468537265885,1
2015-01-06T09:00:00.000Z,12,201.474513646019,357.664039992693,98.9530966552544,29.636979954256,1420534800,172.66043276671866,433.4833870590161,99.71621421595567,40.34574201481317,18.524056000380057,66.09079901046603,10.207689326608683,5.286744796651506,1
2015-01-06T10:00:00.000Z,12,187.041733528236,286.726135157965,90.7527596781869,42.9863261530089,1420538400,172.77548317281082,432.3093290438077,99.64450657965354,40.36686668791874,18.493997996264177,67.11981440654777,10.19800822579745,5.270678462279618,1
2015-01-06T11:00:00.000Z,12,202.598790768568,403.489557519059,103.906708583227,32.7548467429657,1420542000,173.01217609023743,432.0806006983732,99.67833357968189,40.306453831212764,18.610499828668996,66.90008093222548,10.164229154566556,5.293172480780672,1
2015-01-06T12:00:00.000Z,12,183.710404425362,402.575354384718,91.2453990789537,42.4872042175233,1420545600,173.09641410862426,431.8482759242499,99.61193252062105,40.323625094097096,18.56079433424084,66.68549188688279,10.151432269275753,5.275676132039028,1
2015-01-06T13:00:00.000Z,12,201.012356417806,509.35066039447,106.590071825862,29.2520519741914,1420549200,173.31450740791473,432.4537633029235,99.66644923394324,40.23712842909783,18.651507809077685,66.77474186742575,10.130181271657237,5.345208809897986,1
2015-01-06T14:00:00.000Z,12,184.786149594842,380.424767029521,103.044054766786,42.2480290627504,1420552800,173.40343486672813,432.0504377504165,99.69263222256993,40.25271680610288,18.605942192111236,66.67095232049263,10.094913827293214,5.327231077357717,1
2015-01-06T15:00:00.000Z,12,201.970102378847,385.852304563111,102.864839991077,35.6165739184958,1420556400,173.6231784629752,431.69506749512954,99.71703382078921,40.21705416850591,18.702268654013118,66.5355231304177,10.05955829870395,5.3220985293885805,1


In [0]:
# Q6: Train Random Forest binary classifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Convert Spark DataFrame to Pandas
pdf = target.toPandas()

# Select 7-day telemetry features
feature_columns = [
    "avg_volt_7d",
    "avg_rotate_7d",
    "avg_pressure_7d",
    "avg_vibration_7d",
    "std_volt_7d",
    "std_rotate_7d",
    "std_pressure_7d",
    "std_vibration_7d"
]

X = pdf[feature_columns]
y = pdf["failure_next_24h"]

# Check class distribution
print("===== Class Distribution =====")
print(y.value_counts())

# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

# Create Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Train model
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# Evaluate model
print("\n===== Classification Report =====")
print(classification_report(y_test, predictions))

print("===== Confusion Matrix =====")
print(confusion_matrix(y_test, predictions))

===== Class Distribution =====
failure_next_24h
0    858835
1     17165
Name: count, dtype: int64
Training records: 700800
Testing records: 175200

===== Classification Report =====
              precision    recall  f1-score   support

           0       0.99      1.00      1.00    171767
           1       0.99      0.53      0.69      3433

    accuracy                           0.99    175200
   macro avg       0.99      0.77      0.84    175200
weighted avg       0.99      0.99      0.99    175200

===== Confusion Matrix =====
[[171754     13]
 [  1608   1825]]


In [0]:
# Q6: Predict probability of failure within next 24 hours

# Calculate failure probability
pdf["failure_probability_24h"] = model.predict_proba(
    pdf[feature_columns]
)[:, 1]

# Create risk prediction
pdf["failure_prediction"] = pdf[
    "failure_probability_24h"
].apply(
    lambda x: "FAILURE_RISK" if x >= 0.5 else "NORMAL"
)

# Select final results
result = pdf[
    [
        "datetime",
        "machineID",
        "failure_probability_24h",
        "failure_prediction"
    ]
]

# Convert to Spark DataFrame
result_spark = spark.createDataFrame(result)

# Save Q6 result to Gold
result_spark.write.format("delta").mode("overwrite").save(
    f"{gold_path}/failure_probability_24h"
)

print("===== Top Failure Risk Predictions =====")

display(
    result_spark
    .orderBy(
        F.col("failure_probability_24h").desc()
    )
    .limit(20)
)

===== Top Failure Risk Predictions =====


datetime,machineID,failure_probability_24h,failure_prediction
2015-03-20T16:00:00.000Z,95,1.0,FAILURE_RISK
2015-01-04T23:00:00.000Z,84,1.0,FAILURE_RISK
2015-04-19T22:00:00.000Z,69,1.0,FAILURE_RISK
2015-10-29T19:00:00.000Z,99,1.0,FAILURE_RISK
2015-11-18T23:00:00.000Z,86,1.0,FAILURE_RISK
2015-11-16T20:00:00.000Z,42,1.0,FAILURE_RISK
2015-04-19T17:00:00.000Z,92,1.0,FAILURE_RISK
2015-04-10T11:00:00.000Z,13,1.0,FAILURE_RISK
2015-08-18T18:00:00.000Z,26,1.0,FAILURE_RISK
2015-11-16T23:00:00.000Z,42,1.0,FAILURE_RISK


**7. Which specific component (e.g., component 1, 2, 3, or 4) is most likely to fail next?**

In [0]:
# Q7: Prepare features for component failure prediction

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

telemetry = spark.read.format("delta").load(
    f"{silver_path}/PdM_telemetry"
)

failures = spark.read.format("delta").load(
    f"{silver_path}/PdM_failures"
)

# Convert timestamp to seconds
telemetry = telemetry.withColumn(
    "timestamp_long",
    F.col("datetime").cast("long")
)

# 7-day rolling window
window_7d = (
    Window
    .partitionBy("machineID")
    .orderBy("timestamp_long")
    .rangeBetween(-7 * 24 * 3600, 0)
)

# Create telemetry features
features = (
    telemetry
    .withColumn("avg_volt_7d", F.avg("volt").over(window_7d))
    .withColumn("avg_rotate_7d", F.avg("rotate").over(window_7d))
    .withColumn("avg_pressure_7d", F.avg("pressure").over(window_7d))
    .withColumn("avg_vibration_7d", F.avg("vibration").over(window_7d))
    .withColumn("std_volt_7d", F.stddev("volt").over(window_7d))
    .withColumn("std_rotate_7d", F.stddev("rotate").over(window_7d))
    .withColumn("std_pressure_7d", F.stddev("pressure").over(window_7d))
    .withColumn("std_vibration_7d", F.stddev("vibration").over(window_7d))
)

# Find component failures within next 24 hours
component_target = (
    features.alias("t")
    .join(
        failures.alias("f"),
        (F.col("t.machineID") == F.col("f.machineID")) &
        (F.col("f.datetime") > F.col("t.datetime")),
        "inner"
    )
    .withColumn(
        "hours_to_failure",
        (
            F.col("f.datetime").cast("long") -
            F.col("t.datetime").cast("long")
        ) / 3600
    )
    .filter(F.col("hours_to_failure") <= 24)
    .select(
        "t.*",
        F.col("f.failure").alias("failure_component"),
        "hours_to_failure"
    )
    .dropDuplicates(["machineID", "datetime"])
    .dropna()
)

print("Total component failure records:", component_target.count())

print("===== Component Distribution =====")

display(
    component_target
    .groupBy("failure_component")
    .count()
    .orderBy("failure_component")
)

Total component failure records: 17165
===== Component Distribution =====


failure_component,count
comp1,4262
comp2,5964
comp3,2967
comp4,3972


In [0]:
# Q7: Train multiclass Random Forest model

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Convert to Pandas
pdf = component_target.toPandas()

# Select telemetry features
feature_columns = [
    "avg_volt_7d",
    "avg_rotate_7d",
    "avg_pressure_7d",
    "avg_vibration_7d",
    "std_volt_7d",
    "std_rotate_7d",
    "std_pressure_7d",
    "std_vibration_7d"
]

X = pdf[feature_columns]
y = pdf["failure_component"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

# Create multiclass model
model_q7 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Train model
model_q7.fit(X_train, y_train)

# Predict components
predictions = model_q7.predict(X_test)

# Evaluate model
print("===== Component Classification Report =====")
print(
    classification_report(
        y_test,
        predictions
    )
)

print("===== Confusion Matrix =====")
print(
    confusion_matrix(
        y_test,
        predictions
    )
)

Training records: 13732
Testing records: 3433
===== Component Classification Report =====
              precision    recall  f1-score   support

       comp1       1.00      1.00      1.00       852
       comp2       1.00      1.00      1.00      1193
       comp3       1.00      1.00      1.00       593
       comp4       1.00      1.00      1.00       795

    accuracy                           1.00      3433
   macro avg       1.00      1.00      1.00      3433
weighted avg       1.00      1.00      1.00      3433

===== Confusion Matrix =====
[[ 849    2    0    1]
 [   2 1190    0    1]
 [   0    2  591    0]
 [   0    0    0  795]]


In [0]:
# Q7: Predict the most likely component to fail next

# Predict component probabilities
probabilities = model_q7.predict_proba(
    pdf[feature_columns]
)

# Get predicted component
pdf["predicted_component"] = model_q7.predict(
    pdf[feature_columns]
)

# Get highest probability
pdf["failure_probability"] = probabilities.max(axis=1)

# Create final result
result = pdf[
    [
        "datetime",
        "machineID",
        "predicted_component",
        "failure_probability"
    ]
]

# Convert to Spark DataFrame
result_spark = spark.createDataFrame(result)

# Save to Gold
result_spark.write.format("delta").mode("overwrite").save(
    f"{gold_path}/next_component_failure_prediction"
)

print("===== Most Likely Component Failures =====")

display(
    result_spark
    .orderBy(
        F.col("failure_probability").desc()
    )
    .limit(20)
)

===== Most Likely Component Failures =====


datetime,machineID,predicted_component,failure_probability
2015-03-22T17:00:00.000Z,12,comp2,1.0
2015-03-22T09:00:00.000Z,12,comp2,1.0
2015-03-22T08:00:00.000Z,12,comp2,1.0
2015-03-22T06:00:00.000Z,12,comp2,1.0
2015-01-06T11:00:00.000Z,12,comp2,1.0
2015-01-06T12:00:00.000Z,12,comp2,1.0
2015-03-22T12:00:00.000Z,12,comp2,1.0
2015-03-22T11:00:00.000Z,12,comp2,1.0
2015-03-22T19:00:00.000Z,12,comp2,1.0
2015-03-22T14:00:00.000Z,12,comp2,1.0


**8. What is the Remaining Useful Life (RUL) of an active component after a maintenance event?**


In [0]:
# Q8: Prepare Remaining Useful Life features

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

telemetry = spark.read.format("delta").load(
    f"{silver_path}/PdM_telemetry"
)

failures = spark.read.format("delta").load(
    f"{silver_path}/PdM_failures"
)

maint = spark.read.format("delta").load(
    f"{silver_path}/PdM_maint"
)

# Convert timestamp to seconds
telemetry = telemetry.withColumn(
    "timestamp_long",
    F.col("datetime").cast("long")
)

# 7-day rolling window
window_7d = (
    Window
    .partitionBy("machineID")
    .orderBy("timestamp_long")
    .rangeBetween(-7 * 24 * 3600, 0)
)

# Create telemetry features
features = (
    telemetry
    .withColumn("avg_volt_7d", F.avg("volt").over(window_7d))
    .withColumn("avg_rotate_7d", F.avg("rotate").over(window_7d))
    .withColumn("avg_pressure_7d", F.avg("pressure").over(window_7d))
    .withColumn("avg_vibration_7d", F.avg("vibration").over(window_7d))
    .withColumn("std_volt_7d", F.stddev("volt").over(window_7d))
    .withColumn("std_rotate_7d", F.stddev("rotate").over(window_7d))
    .withColumn("std_pressure_7d", F.stddev("pressure").over(window_7d))
    .withColumn("std_vibration_7d", F.stddev("vibration").over(window_7d))
)

# Find next failure
rul_data = (
    features.alias("t")
    .join(
        failures.alias("f"),
        (F.col("t.machineID") == F.col("f.machineID")) &
        (F.col("f.datetime") > F.col("t.datetime")),
        "inner"
    )
    .withColumn(
        "hours_to_failure",
        (
            F.col("f.datetime").cast("long") -
            F.col("t.datetime").cast("long")
        ) / 3600
    )
    .filter(F.col("hours_to_failure") > 0)
    .select(
        F.col("t.machineID").alias("machineID"),
        F.col("t.datetime").alias("datetime"),
        F.col("t.volt").alias("volt"),
        F.col("t.rotate").alias("rotate"),
        F.col("t.pressure").alias("pressure"),
        F.col("t.vibration").alias("vibration"),
        F.col("t.avg_volt_7d").alias("avg_volt_7d"),
        F.col("t.avg_rotate_7d").alias("avg_rotate_7d"),
        F.col("t.avg_pressure_7d").alias("avg_pressure_7d"),
        F.col("t.avg_vibration_7d").alias("avg_vibration_7d"),
        F.col("t.std_volt_7d").alias("std_volt_7d"),
        F.col("t.std_rotate_7d").alias("std_rotate_7d"),
        F.col("t.std_pressure_7d").alias("std_pressure_7d"),
        F.col("t.std_vibration_7d").alias("std_vibration_7d"),
        F.col("hours_to_failure")
    )
)

# Find most recent maintenance before each observation
maintenance_data = (
    rul_data.alias("r")
    .join(
        maint.alias("m"),
        (F.col("r.machineID") == F.col("m.machineID")) &
        (F.col("m.datetime") <= F.col("r.datetime")),
        "left"
    )
    .select(
        F.col("r.*"),
        F.col("m.datetime").alias("maintenance_time")
    )
    .groupBy(
        "machineID",
        "datetime",
        "volt",
        "rotate",
        "pressure",
        "vibration",
        "avg_volt_7d",
        "avg_rotate_7d",
        "avg_pressure_7d",
        "avg_vibration_7d",
        "std_volt_7d",
        "std_rotate_7d",
        "std_pressure_7d",
        "std_vibration_7d",
        "hours_to_failure"
    )
    .agg(
        F.max("maintenance_time").alias("last_maintenance")
    )
    .withColumn(
        "hours_since_maintenance",
        (
            F.col("datetime").cast("long") -
            F.col("last_maintenance").cast("long")
        ) / 3600
    )
    .dropna()
    .dropDuplicates(["machineID", "datetime"])
)

print("RUL records:", maintenance_data.count())

display(maintenance_data.limit(20))

RUL records: 755158


machineID,datetime,volt,rotate,pressure,vibration,avg_volt_7d,avg_rotate_7d,avg_pressure_7d,avg_vibration_7d,std_volt_7d,std_rotate_7d,std_pressure_7d,std_vibration_7d,hours_to_failure,last_maintenance,hours_since_maintenance
12,2015-02-21T06:00:00.000Z,200.72106592893,449.801515816683,104.860795040672,34.2568904797493,171.97848636198694,435.4541081429304,98.99615519721392,39.91949032173607,15.186226536675932,61.92120069506909,10.160321840290328,5.058991581952165,5400.0,2015-02-21T06:00:00.000Z,0.0
12,2015-02-21T07:00:00.000Z,184.262562599914,413.632854079185,119.033013727549,40.2907717735805,172.04006453565484,435.43780342238534,99.20043258430343,39.890418641193186,15.21495607455787,61.92661340274349,10.214246058431895,5.042533778738048,5399.0,2015-02-21T06:00:00.000Z,1.0
12,2015-02-21T08:00:00.000Z,144.237980382354,453.28998169506,115.004480868438,41.1958491672743,172.0204260078193,434.9490600556942,99.27598831052806,39.850124845417945,15.248849720475357,61.45324793985757,10.28382032118505,5.004364694302161,5398.0,2015-02-21T06:00:00.000Z,2.0
12,2015-02-21T09:00:00.000Z,172.782930677065,544.187930521858,90.2085172934445,39.4905159442303,172.01643553388567,435.2434887176227,99.16797437602224,39.821290020018914,15.248559698900252,61.857788226408005,10.282618776832795,4.992228019139576,5397.0,2015-02-21T06:00:00.000Z,3.0
12,2015-02-21T10:00:00.000Z,179.648352937784,368.292212508817,107.016258550012,38.5924586394035,172.12940845693407,434.47625241962953,99.24587807566388,39.87456558228141,15.23386466276532,61.879446988068885,10.291961884076558,4.930035217811554,5396.0,2015-02-21T06:00:00.000Z,4.0
12,2015-02-21T11:00:00.000Z,165.656973788205,509.503871233665,102.317775149646,37.4810832452184,172.22728705739007,434.93736113227413,99.3206249899872,39.83875224209968,15.137956708443607,62.147476447048234,10.267958987524098,4.925278249133969,5395.0,2015-02-21T06:00:00.000Z,5.0
12,2015-02-21T12:00:00.000Z,190.5199510525,419.486550988324,107.714458266158,35.7695600858921,172.34503127912697,435.0424986864636,99.40654030976008,39.78820491239033,15.202638829193766,62.1059596153552,10.277138232433039,4.922930742694195,5394.0,2015-02-21T06:00:00.000Z,6.0
12,2015-02-21T13:00:00.000Z,141.979576413835,494.073580569399,112.38058461043,36.6791888507407,172.0639015725236,435.5417089318702,99.49151703598001,39.76641394782903,15.3225120965507,62.24002616132255,10.324862918383038,4.928523734547843,5393.0,2015-02-21T06:00:00.000Z,7.0
12,2015-02-21T14:00:00.000Z,180.790875625986,471.584681838136,88.6825668770985,38.6049616436361,172.1554234599844,435.8828461998935,99.40166454024983,39.768721482669974,15.328204858752304,62.27886469164091,10.352589964111191,4.92788428464271,5392.0,2015-02-21T06:00:00.000Z,8.0
12,2015-02-21T15:00:00.000Z,178.981053761066,473.612935215797,104.137475465607,42.3412842903527,172.09872748052078,435.5473551171236,99.44740400202754,39.84357647404189,15.284815312307888,61.918833638526365,10.356357991879344,4.869625089929426,5391.0,2015-02-21T06:00:00.000Z,9.0


In [0]:
# Q8: Train Random Forest regression model for RUL

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Convert Spark DataFrame to Pandas
pdf = maintenance_data.toPandas()

# Select features
feature_columns = [
    "avg_volt_7d",
    "avg_rotate_7d",
    "avg_pressure_7d",
    "avg_vibration_7d",
    "std_volt_7d",
    "std_rotate_7d",
    "std_pressure_7d",
    "std_vibration_7d",
    "hours_since_maintenance"
]

X = pdf[feature_columns]
y = pdf["hours_to_failure"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

# Create regression model
model_q8 = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model
model_q8.fit(X_train, y_train)

# Predict RUL
predictions = model_q8.predict(X_test)

# Evaluate model
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("===== RUL Model Results =====")
print("MAE:", round(mae, 2), "hours")
print("RMSE:", round(rmse, 2), "hours")

Training records: 604126
Testing records: 151032
===== RUL Model Results =====
MAE: 392.06 hours
RMSE: 596.81 hours


In [0]:
# Q8: Predict Remaining Useful Life

# Predict RUL for all records
pdf["predicted_rul_hours"] = model_q8.predict(
    pdf[feature_columns]
)

# Create final result
result = pdf[
    [
        "datetime",
        "machineID",
        "hours_since_maintenance",
        "hours_to_failure",
        "predicted_rul_hours"
    ]
]

# Convert to Spark DataFrame
result_spark = spark.createDataFrame(result)

# Save to Gold
result_spark.write.format("delta").mode("overwrite").save(
    f"{gold_path}/remaining_useful_life"
)

print("===== RUL Predictions =====")

display(
    result_spark
    .orderBy("machineID", "datetime")
    .limit(20)
)

===== RUL Predictions =====


datetime,machineID,hours_since_maintenance,hours_to_failure,predicted_rul_hours
2015-01-01T07:00:00.000Z,1,457.0,4055.0,4390.93
2015-01-01T08:00:00.000Z,1,458.0,4054.0,2176.77
2015-01-01T09:00:00.000Z,1,459.0,4053.0,4810.31
2015-01-01T10:00:00.000Z,1,460.0,4052.0,2880.19
2015-01-01T11:00:00.000Z,1,461.0,4051.0,3552.64
2015-01-01T12:00:00.000Z,1,462.0,4050.0,3199.38
2015-01-01T13:00:00.000Z,1,463.0,4049.0,3407.39
2015-01-01T14:00:00.000Z,1,464.0,4048.0,3884.08
2015-01-01T15:00:00.000Z,1,465.0,4047.0,4185.12
2015-01-01T16:00:00.000Z,1,466.0,4046.0,4176.89


**# **9. Are our scheduled preventive maintenance events occurring too early, or are they successfully preventing failures?****

In [0]:
# Q9: Analyze preventive maintenance and failures

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

maint = spark.read.format("delta").load(
    f"{silver_path}/PdM_maint"
)

failures = spark.read.format("delta").load(
    f"{silver_path}/PdM_failures"
)

# Find failures after each maintenance
maintenance_failure = (
    maint.alias("m")
    .join(
        failures.alias("f"),
        (F.col("m.machineID") == F.col("f.machineID")) &
        (F.col("f.datetime") > F.col("m.datetime")),
        "left"
    )
    .withColumn(
        "hours_to_failure",
        (
            F.col("f.datetime").cast("long") -
            F.col("m.datetime").cast("long")
        ) / 3600
    )
    .filter(
        F.col("hours_to_failure").isNull() |
        (F.col("hours_to_failure") > 0)
    )
)

# Select the first failure after each maintenance
window = (
    Window
    .partitionBy(
        F.col("m.machineID"),
        F.col("m.datetime"),
        F.col("m.comp")
    )
    .orderBy(F.col("f.datetime"))
)

result = (
    maintenance_failure
    .withColumn("failure_rank", F.row_number().over(window))
    .filter(F.col("failure_rank") == 1)
    .select(
        F.col("m.machineID").alias("machineID"),
        F.col("m.datetime").alias("maintenance_time"),
        F.col("m.comp").alias("maintained_component"),
        F.col("f.datetime").alias("failure_time"),
        F.col("f.failure").alias("failed_component"),
        "hours_to_failure"
    )
)

print("===== Maintenance vs Next Failure =====")
display(result.limit(20))

===== Maintenance vs Next Failure =====


machineID,maintenance_time,maintained_component,failure_time,failed_component,hours_to_failure
1,2014-06-01T06:00:00.000Z,comp2,2015-01-05T06:00:00.000Z,comp4,5232.0
1,2014-07-16T06:00:00.000Z,comp4,2015-01-05T06:00:00.000Z,comp4,4152.0
1,2014-07-31T06:00:00.000Z,comp3,2015-01-05T06:00:00.000Z,comp4,3792.0
1,2014-12-13T06:00:00.000Z,comp1,2015-01-05T06:00:00.000Z,comp4,552.0
1,2015-01-05T06:00:00.000Z,comp1,2015-03-06T06:00:00.000Z,comp1,1440.0
1,2015-01-05T06:00:00.000Z,comp4,2015-03-06T06:00:00.000Z,comp1,1440.0
1,2015-01-20T06:00:00.000Z,comp1,2015-03-06T06:00:00.000Z,comp1,1080.0
1,2015-01-20T06:00:00.000Z,comp3,2015-03-06T06:00:00.000Z,comp1,1080.0
1,2015-02-04T06:00:00.000Z,comp3,2015-03-06T06:00:00.000Z,comp1,720.0
1,2015-02-04T06:00:00.000Z,comp4,2015-03-06T06:00:00.000Z,comp1,720.0


In [0]:
# Q9: Classify preventive maintenance effectiveness

result_summary = (
    result
    .withColumn(
        "maintenance_effect",
        F.when(
            F.col("hours_to_failure") <= 72,
            "Failure soon after maintenance"
        )
        .when(
            F.col("hours_to_failure") <= 720,
            "Failure within 30 days"
        )
        .otherwise(
            "Longer interval before failure"
        )
    )
)

print("===== Maintenance Effect Summary =====")

display(
    result_summary
    .groupBy("maintenance_effect")
    .agg(
        F.count("*").alias("maintenance_events"),
        F.round(
            F.avg("hours_to_failure"), 2
        ).alias("avg_hours_to_failure")
    )
    .orderBy("avg_hours_to_failure")
)

===== Maintenance Effect Summary =====


maintenance_effect,maintenance_events,avg_hours_to_failure
Failure soon after maintenance,4,21.0
Failure within 30 days,1279,519.26
Longer interval before failure,2003,2279.88


In [0]:
# Q9: Save preventive maintenance analysis

result_summary.write.format("delta").mode("overwrite").save(
    f"{gold_path}/preventive_maintenance_analysis"
)

display(
    result_summary
    .orderBy("hours_to_failure")
    .limit(20)
)

machineID,maintenance_time,maintained_component,failure_time,failed_component,hours_to_failure,maintenance_effect
5,2015-11-20T06:00:00.000Z,comp1,null,null,null,Longer interval before failure
4,2015-12-28T06:00:00.000Z,comp3,null,null,null,Longer interval before failure
4,2015-10-29T06:00:00.000Z,comp3,null,null,null,Longer interval before failure
6,2014-06-01T06:00:00.000Z,comp2,null,null,null,Longer interval before failure
4,2015-11-28T06:00:00.000Z,comp1,null,null,null,Longer interval before failure
4,2015-12-13T06:00:00.000Z,comp4,null,null,null,Longer interval before failure
3,2015-12-18T06:00:00.000Z,comp2,null,null,null,Longer interval before failure
4,2015-10-29T06:00:00.000Z,comp2,null,null,null,Longer interval before failure
5,2015-12-20T06:00:00.000Z,comp3,null,null,null,Longer interval before failure
4,2015-11-13T06:00:00.000Z,comp2,null,null,null,Longer interval before failure


**10. How do different maintenance teams or historical repair types affect the "Mean Time Between Failures" (MTBF) for specific machine models?**


In [0]:
# Q10: Calculate MTBF between consecutive failures

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_path = "/Volumes/workspace/default/t_series_forecasting_data/silver_data"
gold_path = "/Volumes/workspace/default/t_series_forecasting_data/gold_data"

failures = spark.read.format("delta").load(
    f"{silver_path}/PdM_failures"
)

machines = spark.read.format("delta").load(
    f"{silver_path}/PdM_machines"
)

# Order failures for each machine
window = (
    Window
    .partitionBy("machineID")
    .orderBy("datetime")
)

# Find previous failure
failure_intervals = (
    failures
    .withColumn(
        "previous_failure",
        F.lag("datetime").over(window)
    )
    .withColumn(
        "hours_between_failures",
        (
            F.col("datetime").cast("long") -
            F.col("previous_failure").cast("long")
        ) / 3600
    )
    .filter(F.col("previous_failure").isNotNull())
)

# Add machine model
failure_intervals = (
    failure_intervals
    .join(
        machines.select("machineID", "model"),
        "machineID",
        "left"
    )
)

print("===== Failure Intervals =====")
display(failure_intervals.limit(20))

===== Failure Intervals =====


machineID,datetime,failure,previous_failure,hours_between_failures,model
1,2015-03-06T06:00:00.000Z,comp1,2015-01-05T06:00:00.000Z,1440.0,model3
1,2015-04-20T06:00:00.000Z,comp2,2015-03-06T06:00:00.000Z,1080.0,model3
1,2015-06-19T06:00:00.000Z,comp4,2015-04-20T06:00:00.000Z,1440.0,model3
1,2015-09-02T06:00:00.000Z,comp4,2015-06-19T06:00:00.000Z,1800.0,model3
1,2015-10-17T06:00:00.000Z,comp2,2015-09-02T06:00:00.000Z,1080.0,model3
1,2015-12-16T06:00:00.000Z,comp4,2015-10-17T06:00:00.000Z,1440.0,model3
2,2015-03-19T06:00:00.000Z,comp1,2015-03-19T06:00:00.000Z,0.0,model4
2,2015-04-18T06:00:00.000Z,comp2,2015-03-19T06:00:00.000Z,720.0,model4
2,2015-12-29T06:00:00.000Z,comp2,2015-04-18T06:00:00.000Z,6120.0,model4
3,2015-02-06T06:00:00.000Z,comp1,2015-01-07T06:00:00.000Z,720.0,model3


In [0]:
# Q10: Calculate MTBF by machine model and failed component

# Remove zero-hour failure intervals
valid_intervals = failure_intervals.filter(
    F.col("hours_between_failures") > 0
)

# MTBF by machine model
mtbf_by_model = (
    valid_intervals
    .groupBy("model")
    .agg(
        F.count("*").alias("failure_intervals"),
        F.round(
            F.avg("hours_between_failures"), 2
        ).alias("mtbf_hours")
    )
    .orderBy("model")
)

# MTBF by failed component
mtbf_by_component = (
    valid_intervals
    .groupBy("failure")
    .agg(
        F.count("*").alias("failure_intervals"),
        F.round(
            F.avg("hours_between_failures"), 2
        ).alias("mtbf_hours")
    )
    .orderBy("failure")
)

print("===== MTBF by Machine Model =====")
display(mtbf_by_model)

print("===== MTBF by Failed Component =====")
display(mtbf_by_component)

===== MTBF by Machine Model =====


model,failure_intervals,mtbf_hours
model1,159,768.43
model2,143,929.18
model3,178,1265.48
model4,141,1470.34


===== MTBF by Failed Component =====


failure,failure_intervals,mtbf_hours
comp1,147,1214.71
comp2,212,1140.98
comp3,112,963.38
comp4,150,1061.86


In [0]:
# Q10: Save MTBF analysis to Gold

mtbf_by_model.write.format("delta").mode("overwrite").save(
    f"{gold_path}/mtbf_by_model"
)

mtbf_by_component.write.format("delta").mode("overwrite").save(
    f"{gold_path}/mtbf_by_component"
)

print("MTBF analysis saved successfully.")

MTBF analysis saved successfully.
